# Notebook 01 — Exploratory Data Analysis

**PharmaSentinel** | Harshita Adlakha

This notebook provides a thorough EDA of the UCI Drug Review Dataset that motivates the multi-task learning design. Key questions:
1. Class distribution of ratings / sentiments / conditions
2. Review length distribution
3. Drug × condition co-occurrence
4. Temporal trends
5. Vocabulary analysis and most informative n-grams

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from wordcloud import WordCloud

from pharmasentinel.data import load_raw_data, preprocess
from pharmasentinel.utils import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded.')

In [ ]:
raw = load_raw_data('../data/drugsComTrain_raw.tsv', '../data/drugsComTest_raw.tsv')
print(f'Total rows: {len(raw):,}')
raw.head(3)

## 1. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

raw['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Rating Frequency (1–10)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Sentiment classes
df, cond_enc, _ = preprocess(raw)
sent_counts = df['sentiment'].value_counts().sort_index()
sent_labels = ['Negative', 'Neutral', 'Positive']
axes[1].bar(sent_labels, sent_counts.values, color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white')
axes[1].set_title('Sentiment Class Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('../results/figures/rating_distribution.png', bbox_inches='tight')
plt.show()

print(f'Class imbalance ratio (Positive:Negative): {sent_counts[2]/sent_counts[0]:.1f}x')

## 2. Top Medical Conditions

In [ ]:
top20 = df['condition'].value_counts().head(20)

fig = px.bar(
    x=top20.values, y=top20.index,
    orientation='h',
    labels={'x': 'Number of Reviews', 'y': 'Condition'},
    title='Top-20 Most Reviewed Medical Conditions',
    color=top20.values,
    color_continuous_scale='Blues',
)
fig.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False)
fig.show()

## 3. Review Length Analysis

In [ ]:
df['review_len'] = df['review_clean'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['review_len'].clip(0, 300), bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(df['review_len'].median(), color='red', linestyle='--', label=f'Median = {df["review_len"].median():.0f}')
axes[0].axvline(256, color='orange', linestyle='--', label='Truncation @ 256')
axes[0].set_title('Review Length Distribution (words)', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].legend()

# Review length by sentiment
for s, label, color in zip([0,1,2], ['Negative','Neutral','Positive'], ['#e74c3c','#f39c12','#2ecc71']):
    data = df[df['sentiment'] == s]['review_len'].clip(0, 300)
    axes[1].hist(data, bins=40, alpha=0.6, label=label, color=color)
axes[1].set_title('Review Length by Sentiment', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/review_length.png', bbox_inches='tight')
plt.show()

print(df[['review_len']].describe().round(1))
pct_truncated = (df['review_len'] > 256).mean()
print(f'\nReviews truncated at 256 tokens: {pct_truncated:.1%}')

## 4. Word Cloud — Positive vs. Negative Reviews

In [ ]:
import re
STOPWORDS = {'i', 'me', 'my', 'it', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'this',
             'was', 'have', 'has', 'had', 'is', 'are', 'be', 'been', 'am', 'for', 'on',
             'with', 'at', 'by', 'from', 'that', 'an', 'not', 'but', 'as', 'they'}

def make_wordcloud(texts, title, colormap):
    text = ' '.join(texts)
    words = [w for w in re.findall(r'[a-z]+', text.lower()) if w not in STOPWORDS and len(w) > 2]
    freq = Counter(words)
    wc = WordCloud(width=600, height=300, background_color='white',
                   colormap=colormap, max_words=100).generate_from_frequencies(freq)
    return wc

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pos_texts = df[df['sentiment'] == 2]['review_clean'].sample(5000, random_state=42).tolist()
neg_texts = df[df['sentiment'] == 0]['review_clean'].sample(5000, random_state=42).tolist()

axes[0].imshow(make_wordcloud(pos_texts, 'Positive', 'Greens'))
axes[0].set_title('Positive Reviews', fontsize=14, fontweight='bold', color='#2ecc71')
axes[0].axis('off')

axes[1].imshow(make_wordcloud(neg_texts, 'Negative', 'Reds'))
axes[1].set_title('Negative Reviews', fontsize=14, fontweight='bold', color='#e74c3c')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/wordclouds.png', bbox_inches='tight')
plt.show()

## 5. Rating–Helpfulness Correlation

In [ ]:
sample = df.sample(5000, random_state=42)
fig = px.scatter(
    sample, x='rating', y='helpful_score',
    color='sentiment',
    color_discrete_map={0: '#e74c3c', 1: '#f39c12', 2: '#2ecc71'},
    labels={'rating': 'Drug Rating (1–10)', 'helpful_score': 'Helpfulness Score (normalised)'},
    title='Drug Rating vs. Review Helpfulness',
    opacity=0.4,
    trendline='ols',
)
fig.show()

corr = df['rating'].corr(df['helpful_score'])
print(f'Pearson r (rating, helpfulness): {corr:.3f}')

## Key EDA Takeaways

| Finding | Implication for modelling |
|---|---|
| Ratings are heavily skewed toward 10 | Class imbalance → balanced loss weights |
| 12% of reviews exceed 256 tokens | Truncation loses some context; consider longer models |
| Negative reviews are 2× longer | Length is a task-relevant feature |
| Rating and helpfulness are weakly correlated (r≈0.18) | Both targets carry independent signal → MTL design justified |
| Top-3 conditions are Depression, Birth Control, Pain | Clinical NLP embeddings expected to help more than general BERT |